# 00 — What is RAG? Retrieval-Augmented Generation from scratch

Companion notebook to blog post **00 (What is RAG?)**. Builds naive RAG in ~30 lines:
Retrieve → Augment → Generate, over a private handbook no LLM has ever seen.

**Needs an OpenAI API key** (the LLM calls cost well under a cent). Retrieval runs
locally and is free.

The open-book exam metaphor:
1. LLM alone = **closed-book** — memory froze at training time
2. Under pressure, students **invent** (hallucination)
3. **Open the book** — paste the right page into the prompt
4. Someone must **find the right pages** — retrieval, the search problem

In [ ]:
%pip install -q openai fastembed numpy

In [ ]:
import os

def load_key():
    try:                                      # Colab: add OPENAI_API_KEY in the Secrets
        from google.colab import userdata     # sidebar (key icon) and enable notebook access
        return userdata.get("OPENAI_API_KEY")
    except Exception:                         # local Jupyter fallback
        import getpass
        return os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = load_key()

## The closed-book problem, live

Ask about private data: the model refuses — or worse, pressured for a number, it invents
one. (The true answer, from the handbook below, is **48 hours**.)

In [ ]:
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-5.4-mini"

def ask_llm(prompt):
    r = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.choices[0].message.content.strip()

q = "How far in advance must grooming appointments be booked at Sunnyvale Pet Care Center?"
print("closed book:", ask_llm(q))
print()
print("pressured  :", ask_llm(q + " Answer with a specific number of hours only."))

## The private handbook

Eight policies no LLM has seen — that's the point.

In [ ]:
corpus = [
    "Grooming appointments must be booked at least 48 hours in advance",           # doc 0
    "The boarding facility closes at 7 pm on weekdays and 5 pm on weekends",       # doc 1
    "Dogs staying longer than three nights receive a complimentary bath before pickup",  # doc 2
    "Refunds for cancelled boarding are issued within 5 business days",            # doc 3
    "Refunds for cancelled grooming appointments are issued within 10 business days",    # doc 4
    "All pets must have up to date rabies vaccination records on file",            # doc 5
    "Daycare drop off starts at 6:30 am and the last pickup is at 8 pm",           # doc 6
    "A late pickup fee of 15 dollars applies for every 30 minutes after closing",  # doc 7
]

## Step 1 — RETRIEVE: find the right pages

Post 2b's vector search, verbatim: embed the handbook once, embed the question, rank by
cosine. Second demo shows the semantic win — "free bath / long stay" finds
"complimentary bath / longer than three nights" with zero shared words.

In [ ]:
import numpy as np
from fastembed import TextEmbedding

emb_model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
doc_embs = list(emb_model.embed(corpus))      # embed the handbook ONCE

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve(question, top_k=2):
    qe = list(emb_model.embed([question]))[0]
    scored = sorted(((cosine(qe, e), i) for i, e in enumerate(doc_embs)), reverse=True)
    return scored[:top_k]

for question in [q, "Do dogs get a free bath after a long stay?"]:
    print(f"\nQ: {question}")
    for s, i in retrieve(question):
        print(f"  {s:.3f}  doc {i}: {corpus[i]}")

## Step 2 — AUGMENT: the strict prompt

Three jobs: (1) use ONLY the context — answer from the page, not memory;
(2) cite doc ids — every claim traceable; (3) the "I don't know" escape hatch —
without permission to refuse, an LLM handed irrelevant context still answers.

In [ ]:
PROMPT = """You are a precise assistant answering questions about the Sunnyvale Pet Care Center handbook.
Use ONLY the context below. Cite the doc id(s) you used in square brackets, e.g. [doc 3].
If the answer is not in the context, reply exactly: "I don't know based on the provided documents."

Context:
---------
{context}
---------
Question: {question}
Answer:"""

## Step 3 — GENERATE: wire it together

Naive RAG, complete. Watch: the hallucination ("24") becomes "48 hours [doc 0]", the
synonym question works, and the unanswerable question triggers the exact refusal string.

In [ ]:
def rag_answer(question, top_k=2):
    hits = retrieve(question, top_k)                                     # 1. RETRIEVE
    context = "\n".join(f"[doc {i}] {corpus[i]}" for score, i in hits)  # 2. AUGMENT
    return ask_llm(PROMPT.format(context=context, question=question)), hits  # 3. GENERATE

for question in [q,
                 "Do dogs get a free bath after a long stay?",
                 "Does Sunnyvale Pet Care offer cat grooming?"]:
    a, hits = rag_answer(question)
    print(f"\nQ: {question}")
    print("retrieved:", [(round(s, 3), f"doc {i}") for s, i in hits])
    print("A:", a)

## Meeting the villain — retrieval is the bottleneck

The handbook holds both answers: weekends close at **5 pm** [doc 1], late fee **$15 per
30 min** [doc 7]. But "Saturday" ≠ "weekends" and "pick up my dog" smells like the
bath/daycare docs — watch retrieval drift, then watch the pipeline produce a
**confident, cited, WRONG** answer. Every safety feature works; the pages were wrong.

In [ ]:
qv = "Until what time can I pick up my dog on a Saturday, and what happens if I am late?"

print("retrieval top-3:")
for s, i in retrieve(qv, top_k=3):
    print(f"  {s:.3f}  doc {i}: {corpus[i]}")

a, hits = rag_answer(qv, top_k=2)
print("\nRAG answer (top-2 context):")
print("A:", a)
print("\n(True answer: 5 pm on weekends [doc 1], $15 per 30 minutes late [doc 7])")

## PRODUCTION — the LlamaIndex one-liner

Same three steps, industrial parts. `VectorStoreIndex.from_documents` = chunk + embed +
index; `as_query_engine` = retrieve + augment + generate in one object. Note: the
*default* prompt hedges instead of refusing with our exact string — pass a custom
template in production for strict refusals.

In [ ]:
%pip install -q llama-index

In [ ]:
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.openai import OpenAI as LlamaOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = LlamaOpenAI(model="gpt-5.4-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

docs = [Document(text=t, metadata={"doc_id": f"doc {i}"}) for i, t in enumerate(corpus)]

index = VectorStoreIndex.from_documents(docs)       # chunk + embed + index, one line
engine = index.as_query_engine(similarity_top_k=2)  # retrieve + augment + generate

for question in ["How far in advance must grooming appointments be booked?",
                 "Does Sunnyvale Pet Care offer cat grooming?"]:
    r = engine.query(question)
    print("Q:", question)
    print("A:", str(r).strip())
    print("retrieved:", [n.node.metadata["doc_id"] for n in r.source_nodes])
    print()

## Recap — the open-book exam

1. **Closed book** → LLMs can't know your private data
2. **Students invent** → hallucination ("24" vs the true 48)
3. **Open the book** → Augment: strict prompt (only context, cite, may refuse)
4. **Find the right pages** → Retrieve — **the bottleneck**: wrong pages in, confident
   cited wrong answers out

**Next: measuring RAG** — a scoreboard (golden questions, recall, faithfulness) so every
fix has to prove it helps. Posts **2a/2b/2c** crack open the retrieve step itself.